Classificador do arquivo CSV de notícias. 


In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score

# Configurações de download e carga de dados
def download_and_save(url):
    file_name = os.path.basename(url)
    if not os.path.exists(file_name):
        os.system(f"wget {url} -O {file_name}")
    print(f"✅ Arquivo '{file_name}' pronto para uso.")

url_github = "https://raw.githubusercontent.com/ronaldandrade/ebook/07f1cccb41b70488fd000a153139aa814183631a/dados_exemplo/noticias.csv"
caminho_csv = "noticias.csv"
download_and_save(url_github, caminho_csv)

df = pd.read_csv(caminho_csv)
titulos = df['titulo'].tolist()

# Preparação de rótulos (200 primeiras amostras)
amostra_titulos = titulos[:200]
rotulos = [1 if s == 'Positivo' else 0 for s in df['sentimento'][:200]]

# Configuração do Pipeline de Machine Learning
stopwords_pt = ['e', 'em', 'de', 'para', 'com', 'que', 'do', 'da', 'no', 'na', 'o', 'a', 'as', 'os']

pipeline_sentimento = Pipeline([
    ('vetorizador', TfidfVectorizer(max_features=1000, stop_words=stopwords_pt, lowercase=True)),
    ('modelo', SVC(kernel='linear', class_weight='balanced', random_state=42))
])

# Divisão e Validação Estatística
titulos_train, titulos_test, y_train, y_test = train_test_split(
    amostra_titulos, rotulos, test_size=0.2, random_state=42
)

cv_results = cross_validate(pipeline_sentimento, titulos_train, y_train, cv=5, 
                            scoring=['accuracy', 'precision', 'recall'])

# Treinamento e Avaliação Final
pipeline_sentimento.fit(titulos_train, y_train)
y_pred = pipeline_sentimento.predict(titulos_test)

print(f"Acurácia Média (CV): {cv_results['test_accuracy'].mean():.2f} (+/- {cv_results['test_accuracy'].std():.2f})")
print("\n--- Relatório Final no Conjunto de Teste ---")
print(f"Acurácia: {accuracy_score(y_test, y_pred):.2f}")
print(f"Precisão: {precision_score(y_test, y_pred):.2f}")
print(f"Recall: {recall_score(y_test, y_pred):.2f}")

# Inferência: Classificando o dataset completo
df['sentimento_previsto'] = pipeline_sentimento.predict(titulos)
df['sentimento_rotulo'] = ['Positivo' if p == 1 else 'Negativo' for p in df['sentimento_previsto']]

df.to_excel('noticias_classificadas.xlsx', index=False)
print("\n✅ Todas as notícias foram classificadas e salvas em 'noticias_classificadas.xlsx'!")

--2026-03-15 15:42:03--  https://raw.githubusercontent.com/ronaldandrade/ebook/07f1cccb41b70488fd000a153139aa814183631a/dados_exemplo/noticias.csv
Resolvendo raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8000::154, 2606:50c0:8002::154, 2606:50c0:8003::154, ...
Conectando-se a raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8000::154|:443... conectado.
A requisição HTTP foi enviada, aguardando resposta... 200 OK
Tamanho: 2388855 (2,3M) [text/plain]
Salvando em: “noticias.csv”

     0K .......... .......... .......... .......... ..........  2%  262K 9s
    50K .......... .......... .......... .......... ..........  4%  278K 8s
   100K .......... .......... .......... .......... ..........  6%  595K 7s
   150K .......... .......... .......... .......... ..........  8% 1,10M 5s
   200K .......... .......... .......... .......... .......... 10% 1005K 5s
   250K .......... .......... .......... .......... .......... 12%  783K 4s
   300K .......... .......

✅ Arquivo 'noticias.csv' pronto para uso.
Acurácia Média (CV): 0.61 (+/- 0.13)

--- Relatório Final no Conjunto de Teste ---
Acurácia: 0.60
Precisão: 0.58
Recall: 0.70

✅ Todas as notícias foram classificadas e salvas em 'noticias_classificadas.xlsx'!
